In [ ]:
# Cell 1: Setup
!pip install -q sentence-transformers vstash
!git clone https://github.com/stffns/vstash.git /content/vstash 2>/dev/null || (cd /content/vstash && git pull origin develop)
%cd /content/vstash

In [ ]:
# Cell 2: Generate triples (fixed weights, 3 datasets, same as v2)
!PYTHONPATH=/content/vstash python -m experiments.rrf_training_pairs --datasets scifact nfcorpus fiqa

In [ ]:
!hf auth login

In [ ]:
# Cell 3: Train MNRL v5 with EXPLICIT hard negatives
# v2 used: InputExample(texts=[query, positive]) -- negatives only from batch
# v5 uses: InputExample(texts=[query, positive, hard_negative]) -- batch + explicit
import json
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

triples = []
with open("experiments/results/rrf_training_pairs.jsonl") as f:
    for line in f:
        triples.append(json.loads(line))

print(f"Loaded {len(triples)} triples")

# KEY CHANGE: 3 texts per example (query, positive, hard_negative)
examples = [InputExample(texts=[t["query"], t["positive"], t["negative"]]) for t in triples]
loader = DataLoader(examples, shuffle=True, batch_size=64)
loss = losses.MultipleNegativesRankingLoss(model)

print(f"Training: {len(examples)} examples with explicit hard negatives")
print("Epochs: 2, LR: 3e-6, Batch: 64")

model.fit(
    train_objectives=[(loader, loss)],
    epochs=2,
    warmup_steps=50,
    optimizer_params={"lr": 3e-6},
    output_path="experiments/models/vstash-bge-mnrl-v5",
    show_progress_bar=True,
)
model.save("experiments/models/vstash-bge-mnrl-v5")
print("Training done.")

In [ ]:
# Cell 4: Evaluate on all 5 BEIR datasets (full pipeline)
import math
import statistics
import tempfile
from pathlib import Path

from sentence_transformers import SentenceTransformer

import sys

sys.path.insert(0, "/content/vstash")
from experiments.beir_benchmark import download_beir, load_beir
from vstash.store import VstashStore

BASELINES = {
    "scifact": {"BM25": 0.665, "ColBERTv2": 0.693},
    "nfcorpus": {"BM25": 0.325, "ColBERTv2": 0.344},
    "fiqa": {"BM25": 0.236, "ColBERTv2": 0.356},
    "scidocs": {"BM25": 0.158, "ColBERTv2": 0.154},
    "arguana": {"BM25": 0.315, "ColBERTv2": 0.463},
}

MODELS_EVAL = {
    "BGE-small (base)": "BAAI/bge-small-en-v1.5",
    "v2 (batch neg only)": "Stffens/bge-small-rrf-v1",
    "v5 (explicit hard neg)": "experiments/models/vstash-bge-mnrl-v5",
}

DATASETS = ["scifact", "nfcorpus", "fiqa", "scidocs", "arguana"]


def ndcg_at_k(ranked_ids, qr, k=10):
    gains = [qr.get(did, 0) for did in ranked_ids[:k]]
    ideal = sorted(qr.values(), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    dcg = sum(g / math.log2(i + 2) for i, g in enumerate(gains[:k]))
    return dcg / idcg if idcg > 0 else 0.0


all_results = {}
for ds_name in DATASETS:
    cache = download_beir(ds_name)
    corpus, queries, qrels = load_beir(cache)
    test_qids = list(qrels.keys())
    print(f"\n  {ds_name}: {len(corpus)} docs, {len(test_qids)} queries")
    all_results[ds_name] = {}

    for label, model_path in MODELS_EVAL.items():
        st_model = SentenceTransformer(model_path)
        d = st_model.get_sentence_embedding_dimension()
        db_path = tempfile.mktemp(suffix=".db")
        store = VstashStore(db_path, embedding_dim=d)

        doc_ids = list(corpus.keys())
        doc_texts = [
            (corpus[x].get("title", "") + "\n" + corpus[x].get("text", "")).strip()[:512]
            for x in doc_ids
        ]
        all_embs = st_model.encode(doc_texts, show_progress_bar=True, batch_size=256)

        doc_id_map = {}
        batch = []
        for doc_id, text, emb_arr in zip(doc_ids, doc_texts, all_embs):
            path = f"/beir/{doc_id}"
            doc_id_map[path] = doc_id
            batch.append(
                {
                    "path": path,
                    "title": corpus[doc_id].get("title", doc_id),
                    "chunks": [text],
                    "embeddings": [emb_arr.tolist()],
                    "source_type": "text",
                }
            )
            if len(batch) >= 500:
                store.add_documents_batch(batch)
                batch = []
        if batch:
            store.add_documents_batch(batch)

        ndcgs = []
        for qid in test_qids:
            qemb = st_model.encode(queries[qid]).tolist()
            results = store.search(query_embedding=qemb, query_text=queries[qid], top_k=10)
            ranked = [doc_id_map.get(r.path, "") for r in results]
            ndcgs.append(ndcg_at_k(ranked, qrels[qid], 10))

        mean_ndcg = statistics.mean(ndcgs)
        print(f"    {label:>25}: NDCG@10={mean_ndcg:.4f}")
        all_results[ds_name][label] = mean_ndcg
        store.close()
        Path(db_path).unlink(missing_ok=True)

# Summary
print(f"\n{'=' * 90}")
print("  RESULTS: base vs v2 (batch neg) vs v5 (explicit hard neg)")
print(f"{'=' * 90}")
print(
    f"{'Dataset':>12} {'ColBERTv2':>10} {'Base':>10} {'v2':>10} {'v5':>10} {'v5 vs base':>10} {'v5 vs v2':>10}"
)
print(f"  {'-' * 80}")
for ds in DATASETS:
    col = BASELINES[ds]["ColBERTv2"]
    base = all_results[ds].get("BGE-small (base)", 0)
    v2 = all_results[ds].get("v2 (batch neg only)", 0)
    v5 = all_results[ds].get("v5 (explicit hard neg)", 0)
    d_base = (v5 - base) / base * 100 if base > 0 else 0
    d_v2 = (v5 - v2) / v2 * 100 if v2 > 0 else 0
    print(
        f"{ds:>12} {col:>10.3f} {base:>10.4f} {v2:>10.4f} {v5:>10.4f} {d_base:>+9.1f}% {d_v2:>+9.1f}%"
    )

In [ ]:
# Cell 5: Upload to HuggingFace (only if v5 > v2)
!hf auth login
!hf upload Stffens/bge-small-rrf-v2 experiments/models/vstash-bge-mnrl-v5 --commit-message 'v5: MNRL with explicit hard negatives'

In [ ]:
!hf upload Stffens/bge-small-rrf-v2 experiments/models/vstash-bge-mnrl-v5 --commit-message "v5: MNRL with explicit hard negatives, +1-2% vs v2 on all 5 BEIR datasets"

In [ ]:
!cp /content/vstash/experiments/models/README_model_card.md /tmp/README.md && hf upload Stffens/bge-small-rrf-v2 /tmp/README.md --commit-message "Add model card"

In [ ]:
!pip install -q optimum[onnxruntime]

In [ ]:
from optimum.exporters.onnx import main_export

main_export(
    model_name_or_path="experiments/models/vstash-bge-mnrl-v5",
    output="/tmp/onnx_export",
    task="feature-extraction",
)
print("ONNX exported")

In [ ]:
!pip install -q onnxscript

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("experiments/models/vstash-bge-mnrl-v5")
texts = ["test"]
emb = model.encode(texts)
print(f"Model works, dim={len(emb[0])}")

# Export to ONNX manually
dummy = model.tokenize(texts)
dummy = {k: v.to(model.device) for k, v in dummy.items()}

torch.onnx.export(
    model[0].auto_model,
    (dummy["input_ids"], dummy["attention_mask"]),
    "/tmp/model.onnx",
    input_names=["input_ids", "attention_mask"],
    output_names=["last_hidden_state"],
    dynamic_axes={
        "input_ids": {0: "batch", 1: "seq"},
        "attention_mask": {0: "batch", 1: "seq"},
        "last_hidden_state": {0: "batch", 1: "seq"},
    },
    opset_version=14,
)
print("ONNX exported to /tmp/model.onnx")

In [ ]:
!hf upload Stffens/bge-small-rrf-v2 /tmp/model.onnx --commit-message "Add ONNX model"

In [ ]:
!cp /content/vstash/experiments/models/README_model_card.md /tmp/README.md
!hf upload Stffens/bge-small-rrf-v2 /tmp/README.md --commit-message "Update model card with detailed results and methodology"

In [ ]:
!hf upload Stffens/bge-small-rrf-v2 /tmp/README.md --commit-message "Update model card with detailed results"

In [ ]:
with open("/tmp/README.md", "w") as f:
    f.write("""---                                                                                       
language: en                                                                                             
license: apache-2.0                                                                                      
library_name: sentence-transformers                                 
tags:
- sentence-transformers
- embedding
- retrieval
- hybrid-search                                                                                        
- self-supervised
base_model: BAAI/bge-small-en-v1.5                                                                       
pipeline_tag: feature-extraction                                    
---

# bge-small-rrf-v2: A 33M Parameter Model That Beats ColBERTv2 on 3/5 BEIR Datasets                      

**Trained with zero human labels using hybrid retrieval disagreement.**                                  
                                                                    
## Key Result

| Dataset | ColBERTv2 (110M) | BGE-small base (33M) | **This model (33M)** | vs ColBERTv2 |              
|---------|:-:|:-:|:-:|:-:|
| SciFact | 0.693 | 0.646 | **0.695** | **+0.2%** |                                                      
| NFCorpus | 0.344 | 0.330 | **0.395** | **+14.8%** |                                                    
| SciDocs | 0.154 | 0.178 | **0.188** | **+21.8%** |
| FiQA | 0.356 | 0.328 | 0.328 | -7.8% |                                                                 
| ArguAna | 0.463 | 0.419 | 0.424 | -8.4% |                         
                                                                                                        
## Why This Matters                                                 
                                                                                                        
Most embedding improvements need larger models, human labels, or teacher distillation. This model needs  
none. The signal comes from observing where vector search and keyword search disagree. **The system 
improves itself.**                                                                                       
                                                                    
## Training Signal: 82% of queries produce disagreement between vector and keyword search                

- **Vector blind spots** (51%): ranked high by vector but keywords ignore                                
- **Keyword blind spots** (49%): found by keywords but vector misses
                                                                                                        
76K (query, positive, hard_negative) triples. Zero human labels. $0 cost.                                
                                                                                                        
## Training                                                                                              
                                                                    
- Base: BAAI/bge-small-en-v1.5 (33M params, 384d)                                                        
- Loss: MNRL + explicit hard negatives
- TripletLoss destroyed the model (-84%). MNRL preserves knowledge.                                      
- 2 epochs, lr=3e-6, batch 64, ~30 min on T4 GPU                                                         

## Usage                                                                                                 
                                                                    
```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("Stffens/bge-small-rrf-v2")                                                  
embeddings = model.encode(["query", "document"])
                                                                                                        
Train on your own data                                                                                   

pip install vstash sentence-transformers torch                                                           
vstash retrain                                                      
vstash reindex --model ~/.vstash/models/retrained

Links                                                                                                    

- vstash                                                                                                 
- Paper                                                             
- Base model                                                                                             
""")
print("Created /tmp/README.md")

In [ ]:
!hf upload Stffens/bge-small-rrf-v2 /tmp/README.md --commit-message "Update model card"